In [1]:
import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.6.5'

# Process
Starting with the joined timeseries table:
- Filter down to the configurations and time periods of interest
- Add row level calulated fields such as `forecast_leadtime_bin` and `quarter` (one new column), and time series aware calculated fields such as threshold exceedence (one column per threshold).  This requires the joined timeseries uniqueness fields be used. Total number of rows is unchanged.
- Stack (un-pivot) the data such that there is a threshold column that contains the threshold exceeded.  This increses the total number of rows of data, but allows our normal grouping/aggregates to be used.
- Aggregate based on the uniqueness fields plus row level calulated fields and threshold, to determine the min/median/max of each primary and secondary value within each `forecast_leadtime_bin` for each forecast.  This reduces the number of rows and adds a column per aggregation.
- Stack (un-pivot) the data again such that the min/mean/max columns are called primary_value and secondary_value and there is a `window_agg` column to identify which the value is.
- Lastly, we can group by location_id, configuration_name, etc. plus `forecast_leadtime_bin` and `window_agg` and calculate metrics with and without bootstrapping.

# Profiling
- All run with Bleeding Edge 4XL server.
- Desired bootstrap iterations: 1000
- Total locations: 7486

| # Locations | # Days | # Bootstrap Iterations | Spark Params | Aggregation Time | Notes |
| --- | --- | --- | --- | --- | --- |
| 3 | 30 | 10 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 3m34s | |
| 3 | 90 | 10 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 4m41s | |
| 3 | 30 | 100 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 7m17s | |
| 3 | 90 | 100 | Cluster - 64 inst., 16g, 1 core, 2048 part. | 10m36s | |
| All | 90 | 10 | Cluster - 64 inst., 16g, 1 core, 2048 part. | n/a | Job aborted due to stage failure: ShuffleMapStage 30 (first at /srv/conda/envs/notebook/lib/python3.12/site-packages/teehr/querying/utils.py:207) has failed the maximum allowable number of times: 4. |
| All | 90 | 10 | Cluster - 32 inst., 32g, 1 core, 2048 part. | 1hr46m12s | |
| 3 | 30 | 10 | Default | 1m24s | |
| 3 | 90 | 10 | Default | 1m57s | |
| 3 | 30 | 100 | Default | 5m26s | |
| 3 | 90 | 100 | Default | 8m47s | |

In [2]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=32,
    executor_memory="32g",
    executor_cores=2,
    # aws_profile="default"
    update_configs={
        "spark.sql.shuffle.partitions": 2048,
    }
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.11.181
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 32
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 32g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 2
INFO:teehr.evaluat

In [3]:
start = time.perf_counter()

In [4]:
ev = teehr.RemoteReadOnlyEvaluation(spark=spark, enable_spark_proxy=True)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [5]:
joined_cols = ev.table("fcst_joined_timeseries").to_sdf().columns
non_unique_fields = ['primary_value','secondary_value','created_at','updated_at', "value_time"]
uniquenes_fields = [c for c in joined_cols if c not in non_unique_fields]
# uniquenes_fields

INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.


In [6]:
# ids = ev.locations.filter("id like 'usgs-%'").to_sdf().select("id")
# sample = ids.sample(False, 0.01, seed=456).limit(10).collect()
# location_ids = [r.id for r in sample]
# print(location_ids)

In [7]:
filters = [
    TableFilter(
        column="configuration_name",
        operator="=",
        value="nwm30_medium_range"
    ),
    TableFilter(
        column="reference_time",
        operator=">",
        value="2025-01-01T00:00"
    ),
    TableFilter(
        column="reference_time",
        operator="<",
        value="2026-03-31T23:59"
    ),
    # TableFilter(
    #     column="primary_location_id",
    #     operator="in",
    #     value=location_ids
    # )
]

In [8]:
remove_for_quantiles = ["secondary_location_id", "reference_time", "member"]
quantile_group = [c for c in uniquenes_fields if c not in remove_for_quantiles]

calculated_fields = [
    # rcf.GenericSQL(
    #     output_field_name="quarter",
    #     sql_statement="CONCAT('Q', QUARTER(reference_time))"
    # ),
    rcf.ForecastLeadTimeBins(
        bin_size=pd.Timedelta(hours=24),
        output_field_name="forecast_lead_time_bin"
    ),
    tcf.AbovePercentileEventDetection(
        quantile=0.85,
        output_event_field_name="above_q85",
        skip_event_id=True,
        value_field_name="primary_value",
        uniqueness_fields=quantile_group
    ),
    tcf.AbovePercentileEventDetection(
        quantile=0.95,
        output_event_field_name="above_q95",
        skip_event_id=True,
        value_field_name="primary_value",
        uniqueness_fields=quantile_group
    ),
    tcf.AbovePercentileEventDetection(
        quantile=0.99,
        output_event_field_name="above_q99",
        skip_event_id=True,
        value_field_name="primary_value",
        uniqueness_fields=quantile_group
    )
]

In [9]:
# Get raw joined timeseries
tbl = ev.table("fcst_joined_timeseries").filter(filters).add_calculated_fields(calculated_fields)

INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [TableFilter(column='configuration_name', operator=<FilterOperators.eq: '='>, value='nwm30_medium_range'), TableFilter(column='reference_time', operator=<FilterOperators.gt: '>'>, value='2025-01-01T00:00'), TableFilter(column='reference_time', operator=<FilterOperators.lt: '<'>, value='2026-03-31T23:59')].


In [10]:
# len(tbl.distinct_values("primary_location_id"))

7486

In [11]:
# Stack thresholds
threshold_cols = ["above_q85", "above_q95", "above_q99"]
threshold_stack_base_cols = [c for c in tbl.columns if c not in threshold_cols]
# threshold_stack_base_cols

In [12]:
joined_timeseries_with_thresholds_tbl = (
    tbl.selectExpr(
        *threshold_stack_base_cols,
        """
        stack(
            4,
            cast(null as string), true,
            'above_q85', above_q85,
            'above_q95', above_q95,
            'above_q99', above_q99
        ) as (threshold, keep_row)
        """
    )
    .where("keep_row")
    .select(*threshold_stack_base_cols, "threshold")   # no .drop()
)

# print(f"no threshold_rows: {joined_timeseries_with_thresholds_tbl.where("threshold is NULL").count()}")
# print(f"threshold_rows: {joined_timeseries_with_thresholds_tbl.where("threshold is not NULL").count()}")
# print(f"total: {joined_timeseries_with_thresholds_tbl.count()}")


In [13]:
# Add window aggregations
window_metrics = [
    s.Average(
        input_field_names=["primary_value"],
        output_field_name="mean_primary_value"
    ),
    s.Average(
        input_field_names=["secondary_value"],
        output_field_name="mean_secondary_value"
    ),
    s.Minimum(
        input_field_names=["primary_value"],
        output_field_name="min_primary_value"
    ),
    s.Minimum(
        input_field_names=["secondary_value"],
        output_field_name="min_secondary_value"
    ),
    s.Maximum(
        input_field_names=["primary_value"],
        output_field_name="max_primary_value"
    ),
    s.Maximum(
        input_field_names=["secondary_value"],
        output_field_name="max_secondary_value"
    ),
    s.Count(
        input_field_names=["secondary_value"],
        output_field_name="n_in_bin"
    )
]

/tmp/ipykernel_26372/2693311900.py:3: DeprecationWarning: The 'input_field_names' parameter is deprecated and will be removed in a future release. Please use 'primary_field_name' and 'secondary_field_name' instead. For example, instead of input_field_names=['col_a', 'col_b'], use primary_field_name='col_a', secondary_field_name='col_b'.
  s.Average(
/tmp/ipykernel_26372/2693311900.py:7: DeprecationWarning: The 'input_field_names' parameter is deprecated and will be removed in a future release. Please use 'primary_field_name' and 'secondary_field_name' instead. For example, instead of input_field_names=['col_a', 'col_b'], use primary_field_name='col_a', secondary_field_name='col_b'.
  s.Average(
/tmp/ipykernel_26372/2693311900.py:11: DeprecationWarning: The 'input_field_names' parameter is deprecated and will be removed in a future release. Please use 'primary_field_name' and 'secondary_field_name' instead. For example, instead of input_field_names=['col_a', 'col_b'], use primary_field_

In [14]:
group_by_bin = [*uniquenes_fields, "forecast_lead_time_bin","threshold"]
group_by_bin

['reference_time',
 'primary_location_id',
 'secondary_location_id',
 'configuration_name',
 'unit_name',
 'variable_name',
 'member',
 'forecast_lead_time_bin',
 'threshold']

In [15]:
bin_aggs_tbl = joined_timeseries_with_thresholds_tbl.aggregate(
    group_by=group_by_bin,
    metrics=window_metrics
)
# print(f"bin_aggs: {bin_aggs_tbl.count()}")

INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


In [16]:
# after your bin aggregation
# bin_aggs_tbl.select("n_in_bin").summary().show()

In [17]:
pivoted_bin_aggs_tbl = bin_aggs_tbl.selectExpr(
    *group_by_bin,
    """
    stack(
        3,
        'mean', mean_primary_value, mean_secondary_value,
        'min',  min_primary_value,  min_secondary_value,
        'max',  max_primary_value,  max_secondary_value
    ) as (window_agg, primary_value, secondary_value)
    """
)
# print(f"pivoted_bin_aggs: {pivoted_bin_aggs_tbl.count()}")
# pivoted_bin_aggs_tbl.columns

In [18]:
# pivoted_bin_aggs_tbl.show()

In [19]:
# pivoted_bin_aggs_tbl.distinct_values("primary_location_id")
# pivoted_bin_aggs_tbl.distinct_values("reference_time")

In [20]:
# Nash-Sutcliffe efficiency
# Relative mean bias
# Pearson correlation coefficient
# Kling-Gupta efficiency

# Relative mean
# Relative median
# Relative minimum
# Relative maximum
# Relative standard deviation


# Configure bootstrap
bootstap = bs.Stationary(
    reps=10,
    # block_size=100,
    seed=1234,
    quantiles=[0.05, 0.95]
)

In [21]:
metrics = [
    s.Count(),
    s.Average(),
    s.Minimum(),
    s.Maximum(),
    dm.RelativeMean(),
    dm.RelativeMedian(),
    dm.RelativeMinimum(),
    dm.RelativeMaximum(),
    dm.RelativeStandardDeviation(),
    dm.RelativeBias(
        add_epsilon=True,
    ),
    dm.NashSutcliffeEfficiency(
        add_epsilon=True,
    ),
    dm.KlingGuptaEfficiency(
        add_epsilon=True,
    ),
    dm.PearsonCorrelation(
        add_epsilon=True,
    ),
    dm.RelativeMean(
        output_field_name="relative_mean_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.RelativeMedian(
        output_field_name="relative_median_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.RelativeMinimum(
        output_field_name="relative_minimum_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.RelativeMaximum(
        output_field_name="relative_maximum_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.RelativeStandardDeviation(
        output_field_name="relative_standard_deviation_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.NashSutcliffeEfficiency(
        output_field_name="nash_sutcliffe_efficiency_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.RelativeBias(
        output_field_name="relative_bias_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.PearsonCorrelation(
        output_field_name="pearson_correlation_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
    dm.KlingGuptaEfficiency(
        output_field_name="kling_gupta_efficiency_boot",
        bootstrap=bootstap,
        unpack_results=True
    ),
]

In [22]:
group_by = [
    "primary_location_id",
    "secondary_location_id",
    "configuration_name",
    "unit_name",
    "variable_name",
    "member",
    "forecast_lead_time_bin",
    "threshold",
    "window_agg",
]

In [23]:
%%time
results = pivoted_bin_aggs_tbl.aggregate(
    group_by=group_by,
    metrics=metrics
).order_by(group_by)

INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Setting order_by ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'forecast_lead_time_bin', 'threshold', 'window_agg'].


CPU times: user 888 ms, sys: 322 ms, total: 1.21 s
Wall time: 1h 46min 12s


In [24]:
for k in [
    "spark.executor.memory",
    "spark.executor.memoryOverhead",
    "spark.kubernetes.memoryOverheadFactor",
]:
    try:
        print(k, spark.conf.get(k))
    except Exception as e:
        print(k, "not set")

spark.executor.memory 32g
spark.executor.memoryOverhead not set
spark.kubernetes.memoryOverheadFactor not set


In [25]:
# %%time
# results.explain(mode="simple")

In [26]:
# %%time
# results.show()

In [27]:
end = time.perf_counter()

elapsed_seconds = end - start
print(f"{elapsed_seconds:.6f} s")

6542.086844 s


In [28]:
spark.stop()